# Tutorial 3: Binary Radius

A self-contained introduction to the binary exact radius and the multiclass lower bound.

In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "experiments").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from ghosts.radii import compute_rho_out

PI = np.pi

deltas = np.linspace(0.1, 10, 200)
rho_lower = PI / deltas
rho_exact = np.sqrt(deltas**2 + PI**2) / deltas

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(deltas, rho_lower, label=r'Lower bound $\pi/\Delta$', lw=2)
ax.plot(deltas, rho_exact, '--', label=r'Exact (binary) $\sqrt{\delta^2+\pi^2}/\Delta_a$', lw=2)
ax.set_xlabel(r'Logit gap $\Delta$')
ax.set_ylabel(r'Convergence radius $\rho$')
ax.set_title('Convergence radius shrinks as predictions sharpen')
ax.legend()
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()


In [ ]:
uniform = torch.tensor([[0.0, 0.0]])
print(f"Uniform: rho = {compute_rho_out(uniform, gap='maxmin', reduce='mean'):.2f}")

confident = torch.tensor([[0.0, 10.0]])
print(f"Confident: rho = {compute_rho_out(confident, gap='maxmin', reduce='mean'):.4f}")

batch = torch.tensor([[0.0, 1.0], [0.0, 5.0], [0.0, 10.0]])
per_sample = compute_rho_out(batch, gap='maxmin', reduce='per_sample')
print(f"Per-sample rho: {per_sample}")


In [ ]:
def ce_loss(z, target=0):
    return np.log1p(np.exp(-z)) if target == 0 else np.log1p(np.exp(z))

z0 = 2.0
rho = PI / z0

taus = np.linspace(-3 * rho, 3 * rho, 500)
losses = [ce_loss(z0 + t) for t in taus]

L0 = ce_loss(z0)
sig = 1 / (1 + np.exp(-z0))
dL = -(1 - sig)
d2L = sig * (1 - sig)
taylor2 = L0 + dL * taus + 0.5 * d2L * taus**2

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(taus, losses, 'k-', lw=2, label='True loss')
ax.plot(taus, taylor2, 'r--', lw=1.5, label='Quadratic Taylor')
ax.axvline(-rho, color='blue', ls=':', alpha=0.7, label=fr'$\pm\rho = \pm{rho:.2f}$')
ax.axvline(rho, color='blue', ls=':', alpha=0.7)
ax.set_xlabel(r'Step size $\tau$')
ax.set_ylabel('Loss')
ax.set_title(f'Loss landscape along a direction (logit gap = {z0})')
ax.legend()
ax.set_ylim(-0.1, 3)
plt.tight_layout()
plt.show()
